In [ ]:
import os
import yaml
from typing import List
from datetime import datetime


import requests
from playwright.async_api import async_playwright

from shoppinglist_rnd.models import Recipe, IngestionMetadata

In [ ]:
PROJECT_ROOT = os.path.abspath('..')
DATA_FOLDER = os.path.join(PROJECT_ROOT, ".data")

In [ ]:
def fetch_recipes() -> List[Recipe]:
    response = requests.get("https://shoppinglist-api.nilpath.se/recipe")
    response.raise_for_status()
    recipes_data = response.json()
    return [Recipe(**data) for data in recipes_data]


In [ ]:
recipes = fetch_recipes()

In [ ]:
recipes_with_urls = [recipe for recipe in recipes if recipe.url]
len(recipes_with_urls)

In [ ]:
async def fetch_html_from_url(url: str, timeout: int = 5000) -> str:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        try:
            await page.goto(url, timeout=timeout)
            await page.wait_for_load_state("load", timeout=timeout)
            html = await page.content()
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            html = ""
        finally:
            await browser.close()
        
        return html


In [ ]:
async def download_recipes(
    recipes: List[Recipe], 
    cache_folder: str = DATA_FOLDER,
    timeout: int = 5000,
):

    ingestion_time = datetime.utcnow().isoformat()
    ingestion_path = os.path.join(cache_folder, str(ingestion_time))

    metadata = IngestionMetadata(
        timestamp=ingestion_time,
        record_count=len(recipes),
    )

    for recipe in recipes:
        recipe_path = os.path.join(ingestion_path, str(recipe.id))
        metadata.src_paths.append(recipe_path)
        os.makedirs(recipe_path, exist_ok=True)
        
        with open(os.path.join(recipe_path, "recipe.yaml"), "w", encoding="utf-8") as f:
            yaml.dump(recipe.model_dump(mode="json"), f, default_flow_style=False, allow_unicode=True)

        if recipe.url:
            html = await fetch_html_from_url(recipe.url, timeout=timeout)
            with open(os.path.join(recipe_path, "content.html"), "w", encoding="utf-8") as f:
                f.write(html)

    with open(os.path.join(ingestion_path, "metadata.yaml"), "w", encoding="utf-8") as f:
        yaml.dump(metadata.model_dump(mode="json"), f, default_flow_style=False, allow_unicode=True)

In [ ]:
await download_recipes(recipes_with_urls, timeout=1.5 * 60 * 1000)